# 04 — HOG-PCA + SVM

PCA được fit trên HOG train trước chuẩn hóa L2 toàn vector. Ngưỡng 95% là phương sai giữ lại của train, không phải mức Accuracy hay lượng thông tin nhãn được bảo đảm giữ lại.


In [ ]:
# KAGGLE_INPUT_SETUP: tìm code và DATA trong các Dataset đã gắn.
from pathlib import Path
import os

kaggle_input = Path('/kaggle/input')
dataset_roots = []
if kaggle_input.is_dir():
    dataset_roots.extend(kaggle_input.glob('datasets/*/*'))
    dataset_roots.extend(kaggle_input.glob('*'))
dataset_roots = [p for p in dataset_roots if p.is_dir()]

project_hint = os.environ.get('CS231_PROJECT_ROOT')
project_candidates = ([Path(project_hint)] if project_hint else
                      [Path.cwd(), *Path.cwd().parents, *dataset_roots])
project_matches = sorted({p.resolve() for p in project_candidates
                          if (p / 'src/SVM/pipeline.py').is_file()})
if len(project_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset code chứa src/SVM/pipeline.py; '
        'nếu có nhiều bản, đặt CS231_PROJECT_ROOT. '
        f'Tìm thấy: {project_matches}'
    )
project = project_matches[0]

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = ([Path(data_hint)] if data_hint else
                   [project / 'DATA', *(p / 'DATA' for p in dataset_roots)])
data_matches = sorted({p.resolve() for p in data_candidates
                       if (p / 'train/10000').is_dir()})
if len(data_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset DATA chứa DATA/train/10000; '
        'nếu có nhiều bản, đặt CS231_DATA_ROOT. '
        f'Tìm thấy: {data_matches}'
    )
data = data_matches[0]

os.environ['CS231_PROJECT_ROOT'] = str(project)
os.environ['CS231_DATA_ROOT'] = str(data)
print('Code:', project, (project / 'src/SVM/pipeline.py').is_file())
print('DATA:', data, (data / 'train/10000').is_dir())


In [ ]:
PREVIOUS_ARTIFACT = 'hog/svm_grid_dev.csv'
# PREVIOUS_NOTEBOOK_OUTPUT: khôi phục toàn bộ kết quả của notebook ngay trước đó.
from pathlib import Path
from shutil import copytree
import os

if Path('/kaggle/working').is_dir():
    notebook_root = Path('/kaggle/input/notebooks')
    attached = sorted(p for p in notebook_root.glob('*/*') if p.is_dir())
    if len(attached) != 1:
        raise FileNotFoundError(
            'Hãy gắn đúng một Notebook Output của bước ngay trước đó. '
            f'Đang thấy: {attached}'
        )
    configs = [p for p in attached[0].rglob('experiment_config.json')
               if p.parent.name == 'inputs' and p.parent.parent.name == 'results_224']
    if len(configs) != 1:
        raise FileNotFoundError(
            'Notebook Output đã gắn phải chứa results_224/inputs/experiment_config.json. '
            f'Tìm thấy: {configs}'
        )
    source = configs[0].parents[1]
    expected = source / PREVIOUS_ARTIFACT
    if not expected.is_file():
        raise FileNotFoundError(
            f'Output không phải bản chạy thành công của bước ngay trước: thiếu {expected}'
        )
    target = Path(os.environ.get('CS231_SVM_RESULTS_ROOT',
                                 '/kaggle/working/CS231_SVM/results_224'))
    copytree(source, target, dirs_exist_ok=True)
    for relative in ('inputs/train_manifest.csv', 'inputs/dev_manifest.csv'):
        if not (target / relative).is_file():
            raise FileNotFoundError(f'Output thiếu {target / relative}')
    print('Đã chép:', source, '→', target)
else:
    print('Chạy local: dùng results_224 hiện có trong repository.')


In [ ]:
from pathlib import Path
import os, sys

# Có thể chỉ rõ ba đường dẫn bằng biến môi trường khi gắn repository/dataset trên Kaggle.
root_hint = os.environ.get('CS231_PROJECT_ROOT')
root_candidates = [Path(root_hint)] if root_hint else [Path.cwd(), *Path.cwd().parents]
if Path('/kaggle/input').is_dir():
    root_candidates += list(Path('/kaggle/input').glob('*'))
    root_candidates += list(Path('/kaggle/input').glob('*/*'))
PROJECT_ROOT = next((p.resolve() for p in root_candidates
                     if (p / 'src/SVM/pipeline.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy repository. Đặt CS231_PROJECT_ROOT.')
sys.path.insert(0, str(PROJECT_ROOT))

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = [Path(data_hint)] if data_hint else [PROJECT_ROOT / 'DATA']
if not data_hint and Path('/kaggle/input').is_dir():
    data_candidates += list(Path('/kaggle/input').glob('*/DATA'))
    data_candidates += list(Path('/kaggle/input').glob('*/*/DATA'))
DATA_ROOT = next((p.resolve() for p in data_candidates
                  if (p / 'train/10000').is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Không tìm thấy DATA/train/10000. Đặt CS231_DATA_ROOT.')

result_hint = os.environ.get('CS231_SVM_RESULTS_ROOT')
if result_hint:
    RESULTS_ROOT = Path(result_hint).resolve()
elif Path('/kaggle/working').is_dir():
    RESULTS_ROOT = Path('/kaggle/working/CS231_SVM/results_224')
else:
    RESULTS_ROOT = PROJECT_ROOT / 'src/SVM/results_224'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Repository:', PROJECT_ROOT)
print('DATA:', DATA_ROOT)
print('Kết quả:', RESULTS_ROOT)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.SVM.pipeline import (
    CLASSES, FEATURE_FOLDERS, manifest_for_split, extract_matrix, l2,
    save_features, load_features, same_manifest, run_dev_grid, save_dev_result,
    save_json, hsv12, hog_raw,
)
from src.preprocessing import read_rgb224


## 1. Đọc HOG trước L2 và manifest


In [ ]:
HOG_DIR = RESULTS_ROOT / 'hog'
OUTPUT = RESULTS_ROOT / 'hog_pca'
OUTPUT.mkdir(parents=True, exist_ok=True)
FEATURE_ID = 'HOG_PCA'
FEATURE_PIPELINE = 'RGB224 -> gray -> HOG raw -> PCA95 fit train -> whole-vector L2'
X_train_raw = np.load(HOG_DIR / 'X_train_hog_before_l2.npy', mmap_mode='r')
X_dev_raw = np.load(HOG_DIR / 'X_dev_hog_before_l2.npy', mmap_mode='r')
_, y_train, train_manifest = load_features(HOG_DIR, 'train')
_, y_dev, dev_manifest = load_features(HOG_DIR, 'dev')
if X_train_raw.shape != (len(train_manifest), 6084) or X_dev_raw.shape != (len(dev_manifest), 6084):
    raise ValueError('Shape HOG raw không khớp')


## 2. Fit PCA chỉ trên train

Cùng một đối tượng PCA được dùng để transform train và dev. Sau phép chiếu, từng vector được L2 để tính khoảng cách theo cùng quy tắc với nhánh HOG-PCA của KNN.


In [ ]:
import pickle
from time import perf_counter
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95, svd_solver='full', whiten=False)
start = perf_counter()
pca.fit(X_train_raw)
fit_seconds = perf_counter() - start
X_train = l2(pca.transform(X_train_raw))
X_dev = l2(pca.transform(X_dev_raw))
save_features(OUTPUT, train_manifest, dev_manifest, X_train, X_dev)
with (OUTPUT / 'pca.pkl').open('wb') as stream:
    pickle.dump(pca, stream)
print('Số chiều:', X_train_raw.shape[1], '→', X_train.shape[1])
print('Phương sai giữ lại:', pca.explained_variance_ratio_.sum())
print('Giây fit PCA:', fit_seconds)


## 3. Phương sai tích lũy

Đường phương sai chỉ mô tả mức biến thiên của HOG train được giữ sau phép chiếu. Tác dụng đối với phân loại phải đọc từ kết quả dev.


In [ ]:
cumulative = np.cumsum(pca.explained_variance_ratio_)
pd.DataFrame({'component': np.arange(1, len(cumulative) + 1),
              'explained_variance_ratio': pca.explained_variance_ratio_,
              'cumulative_variance': cumulative}).to_csv(OUTPUT / 'pca_variance.csv', index=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(cumulative) + 1), cumulative)
ax.axhline(0.95, color='red', linestyle='--')
ax.set_xlabel('Số thành phần')
ax.set_ylabel('Phương sai tích lũy')
plt.tight_layout()
plt.savefig(OUTPUT / 'pca_variance.png', dpi=150)
plt.show()


## 4. Khảo sát SVC trên dev và đối chiếu HOG gốc


In [ ]:
grid, best_pred, best_params = run_dev_grid(X_train, y_train, X_dev, y_dev)
scores = save_dev_result(OUTPUT, FEATURE_ID, grid, best_pred, dev_manifest)
save_json(OUTPUT / 'run_config.json', {
    'feature_id': FEATURE_ID, 'feature_dim': int(X_train.shape[1]),
    'preprocessing': 'RGB uint8 224x224 OpenCV INTER_LINEAR',
    'feature_pipeline': FEATURE_PIPELINE,
    'svm_grid': 'linear C=(0.1,1,10); rbf C=(1,10), gamma=(scale,10)',
    'class_weight': None, 'cache_size_mb': 512,
    'decision_function_shape': 'ovr', 'break_ties': True,
    'best_params': best_params, 'fit_split': 'train', 'selection_split': 'dev',
})
print('Cấu hình thắng trên dev:')
display(grid.head(7))
print('Accuracy:', scores['accuracy'], 'Macro-F1:', scores['macro_f1'],
      'MAE VND/ảnh:', scores['mae_vnd'])
display(pd.DataFrame(scores['per_class'])[
    ['denomination_vnd', 'support', 'precision', 'recall', 'f1']])


In [ ]:
config = json.loads((OUTPUT / 'run_config.json').read_text(encoding='utf-8'))
config.update({'pca_dim': int(pca.n_components_),
               'explained_variance_ratio_sum': float(pca.explained_variance_ratio_.sum()),
               'pca_fit_seconds': fit_seconds, 'pca_file': 'pca.pkl'})
save_json(OUTPUT / 'run_config.json', config)
baseline = pd.read_csv(HOG_DIR / 'svm_best_dev.csv')
display(pd.concat([baseline, pd.read_csv(OUTPUT / 'svm_best_dev.csv')], ignore_index=True)[
    ['feature_id', 'kernel', 'C', 'gamma', 'accuracy', 'macro_f1', 'mae_vnd']])


## 5. Ma trận nhầm lẫn


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(np.asarray(scores['confusion_matrix']),
                       display_labels=CLASSES).plot(ax=ax, cmap='Blues',
                                                     values_format='d', xticks_rotation=45)
ax.set_title(f'{FEATURE_ID} + SVM — dev')
plt.tight_layout()
plt.savefig(OUTPUT / 'confusion_matrix_dev.png', dpi=180)
plt.show()
